In [3]:
import urllib.parse

conn_str = urllib.parse.quote_plus(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=SANDIPAN\SQLEXPRESS;"
    r"DATABASE=master;"
    r"Trusted_Connection=yes;"
)

%reload_ext sql
%sql mssql+pyodbc:///?odbc_connect={conn_str}

#### Chapter 2: Practice QnA
**Topics:** NULL Handling · Aggregation · GROUP BY · Subqueries · EXISTS / IN / ANY / ALL

**Database:** MSSQL · Tables used: `orders`, `returns`, `employee`

---
#### Q1. Find all orders where city is NULL.

In [5]:
%%sql

SELECT TOP 3 *
FROM orders
WHERE city IS NULL

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
14.0,CA-2020-161389,2020-12-05,2020-12-10,Standard Class,IM-15070,Irene Maddox,Consumer,United States,None,Washington,98103.0,West,OFF-BI-10003656,Office Supplies,Binders,Fellowes PB200 Plastic Comb Binding Machine,407.97600000000006,3.0,0.2,132.59219999999993
24.0,US-2021-156909,2021-07-16,2021-07-18,Second Class,SF-20065,Sandra Flanagan,Consumer,United States,None,Pennsylvania,19140.0,East,FUR-CH-10002774,Furniture,Chairs,"Global Deluxe Stacking Chair, Gray",71.37199999999999,2.0,0.3,-1.0196000000000005


---
#### Q2. Show customer_name. If customer_name is NULL, display 'Unknown Customer' instead.

In [10]:
%%sql

SELECT TOP 3 *,
COALESCE(customer_name,'Unknown Customer') AS customer_name
FROM orders;

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,customer_name_1
1.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2.0,0.0,41.9136,Claire Gute
2.0,CA-2020-152156,2020-11-08,2020-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.9399999999999,3.0,0.0,219.58199999999997,Claire Gute
3.0,CA-2020-138688,2020-06-12,2020-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2.0,0.0,6.8713999999999995,Darrin Van Huff


---
#### Q3. Count total rows in orders, and separately count how many rows have a non-NULL postal_code.

> **Trick:** If these two numbers differ, that difference is exactly how many rows have a missing postal_code.

In [13]:
%%sql

SELECT 
    COUNT(*) AS total_rows_in_orders, 
    COUNT(postal_code) AS total_rows_with_postal_code
FROM orders

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


total_rows_in_orders,total_rows_with_postal_code
9994,9983


---
#### Q4. Find total profit, first order date, and latest order date for each category.

In [15]:
%%sql

SELECT 
    category, 
    ROUND(SUM(profit),1) AS total_profit,
    MIN(order_date) AS first_order_date,
    MAX(order_date) AS last_order_date
FROM orders
GROUP BY category

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


category,total_profit,first_order_date,last_order_date
Office Supplies,122490.8,2018-01-03,2021-12-30
Furniture,18451.3,2018-01-06,2021-12-30
Technology,145454.9,2018-01-06,2021-12-30


---
#### Q5. Find total sales for each region and ship_mode combination, for orders placed in year 2020 only.

In [17]:
%%sql

SELECT 
    region, 
    ship_mode,
    ROUND(SUM(sales),1) AS total_sales
FROM orders
WHERE YEAR(order_date) = 2020
GROUP BY region,ship_mode

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


region,ship_mode,total_sales
West,Second Class,36881.8
East,First Class,25457.5
West,Standard Class,102334.6
Central,Same Day,4980.2
East,Second Class,24284.4
East,Same Day,10017.0
West,First Class,35562.5
Central,Standard Class,105340.9
South,Second Class,30563.5
South,Standard Class,43829.5


---
#### Q6. Find the total number of distinct products sold in each category.

In [20]:
%%sql
SELECT category, COUNT(DISTINCT product_id) AS distinct_products
FROM orders
GROUP BY category

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


category,distinct_products
Furniture,375
Office Supplies,1083
Technology,404


In [23]:
%%sql
SELECT category, COUNT(product_id) AS products
FROM orders
GROUP BY category

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


category,products
Office Supplies,6026
Furniture,2121
Technology,1847


In [24]:
%%sql
SELECT category, COUNT(*) AS products
FROM orders
GROUP BY category

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


category,products
Office Supplies,6026
Furniture,2121
Technology,1847


---
#### Q7. Find the top 5 sub-categories in the West region by total quantity sold.

In [30]:
%%sql

SELECT TOP 5 sub_category,SUM(quantity) AS total_quantity
FROM orders
WHERE region = 'West'
GROUP by sub_category
ORDER BY SUM(quantity) DESC

 * mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DSANDIPAN%5CSQLEXPRESS%3BDATABASE%3Dmaster%3BTrusted_Connection%3Dyes%3B
Done.


sub_category,total_quantity
Binders,1868.0
Paper,1702.0
Furnishings,1175.0
Phones,1068.0
Storage,1039.0


---
#### Q8. Find sub-categories where average profit is greater than half of the GLOBAL maximum profit.

> **Trap:** Do NOT write `HAVING AVG(profit) > MAX(profit) / 2` — that `MAX(profit)` is calculated per sub_category group, not globally.
>
> **Fix:** Use a subquery in HAVING to get the global max separately.

In [ ]:
%%sql
-- WRONG approach (MAX here is per-group, not global)
SELECT sub_category
FROM orders
GROUP BY sub_category
HAVING AVG(profit) > MAX(profit) / 2;

In [ ]:
%%sql
-- CORRECT approach (subquery calculates the global max independently)
SELECT sub_category
FROM orders
GROUP BY sub_category
HAVING AVG(profit) > (SELECT MAX(profit) FROM orders) / 2;

---
## Q9. Find the average sales for each ship_mode.

In [ ]:
%%sql
SELECT
    ship_mode,
    AVG(sales) AS avg_sales
FROM orders
GROUP BY ship_mode;

---
## Q10. Find the maximum profit in each region.

In [ ]:
%%sql
SELECT
    region,
    MAX(profit) AS max_profit
FROM orders
GROUP BY region;

---
## Q11. Find the number of orders placed each year.

In [ ]:
%%sql
SELECT
    YEAR(order_date) AS order_year,
    COUNT(*) AS no_of_orders
FROM orders
GROUP BY YEAR(order_date)
ORDER BY order_year;

---
## Q12. Find employees whose salary is greater than the company-wide average salary. (Scalar Subquery)

In [ ]:
%%sql
SELECT emp_name, salary
FROM employee
WHERE salary > (SELECT AVG(salary) FROM employee);

---
## Q13. Find employees whose salary is greater than the average salary of their OWN department. (Correlated Subquery)

> The inner query re-runs once for each employee row because it references `t1.dept_id` from the outer query.

In [ ]:
%%sql
SELECT emp_name, salary, dept_id
FROM employee t1
WHERE salary > (
    SELECT AVG(salary)
    FROM employee t2
    WHERE t2.dept_id = t1.dept_id
);

---
## Q14. Find all orders that have been returned — using EXISTS.

In [ ]:
%%sql
SELECT o.order_id, o.customer_name, o.sales
FROM orders o
WHERE EXISTS (
    SELECT 1
    FROM returns r
    WHERE r.order_id = o.order_id
);

---
## Q15. Find all orders that have been returned — using IN.

> Same result as Q14, different approach.

In [ ]:
%%sql
SELECT order_id, customer_name, sales
FROM orders
WHERE order_id IN (
    SELECT order_id FROM returns
);

---
## Q16. Find all orders that have NEVER been returned — using NOT EXISTS (safe, recommended).

In [ ]:
%%sql
SELECT o.order_id, o.customer_name, o.sales
FROM orders o
WHERE NOT EXISTS (
    SELECT 1
    FROM returns r
    WHERE r.order_id = o.order_id
);

---
## Q17. Find all orders that have NEVER been returned — using NOT IN with NULL safety.

> **Always add `WHERE order_id IS NOT NULL` inside the subquery when using NOT IN.**
> Without it, if even one NULL exists in returns.order_id, you get zero results silently.

In [ ]:
%%sql
SELECT order_id, customer_name, sales
FROM orders
WHERE order_id NOT IN (
    SELECT order_id FROM returns WHERE order_id IS NOT NULL
);

---
## Q18. Find employees earning more than at least one employee in dept_id = 1. (ANY)

In [ ]:
%%sql
SELECT emp_name, salary
FROM employee
WHERE salary > ANY (
    SELECT salary FROM employee WHERE dept_id = 1
);

---
## Q19. Find employees earning more than every employee in dept_id = 1. (ALL)

In [ ]:
%%sql
SELECT emp_name, salary
FROM employee
WHERE salary > ALL (
    SELECT salary FROM employee WHERE dept_id = 1
);

---
## Q20. Show each customer_name alongside how many orders they placed. (Scalar Subquery in SELECT)

In [ ]:
%%sql
SELECT
    customer_name,
    (SELECT COUNT(*) FROM orders o2 WHERE o2.customer_name = o1.customer_name) AS order_count
FROM orders o1
GROUP BY customer_name;

---
## Q21. Find total quantity sold, average sales, max profit, and min discount — all in one query, grouped by category.

In [ ]:
%%sql
SELECT
    category,
    SUM(quantity)    AS total_quantity,
    AVG(sales)       AS avg_sales,
    MAX(profit)      AS max_profit,
    MIN(discount)    AS min_discount
FROM orders
GROUP BY category;

---
## Q22. Show order_id and a column showing sales divided by quantity (price per unit). Handle the case where quantity might be zero.

In [ ]:
%%sql
-- NULLIF(quantity, 0) returns NULL if quantity = 0
-- Dividing by NULL gives NULL instead of a divide-by-zero error
SELECT
    order_id,
    sales / NULLIF(quantity, 0) AS price_per_unit
FROM orders;

---
## Q23. Show orders where order_id appears in the returns table — using a subquery inside IN. (Multi-row subquery)

In [ ]:
%%sql
SELECT order_id, customer_name, sales
FROM orders
WHERE order_id IN (
    SELECT order_id
    FROM returns
    WHERE return_reason = 'Wrong Items'
);

---
## Bonus: Quick Self-Check

Answer these in your head before moving to Chapter 3:

1. What is the difference between `COUNT(*)` and `COUNT(col)`?
2. What does `NULLIF(col, 0)` do and when do you use it?
3. Why does `NOT IN` silently return zero rows when the subquery has a NULL value?
4. What is the difference between `> ANY` and `> ALL`?
5. When would you use a correlated subquery instead of a JOIN?
6. What does `COALESCE(city, state, 'Unknown')` do if city is NULL but state is not NULL?
7. What is the GROUP BY scoping trap and how do you fix it?
8. What does `SELECT 1` inside an `EXISTS` clause mean — why not `SELECT *`?